## IMDB reviews embeddings

Perform vectorization when the input is in the form of `tensorflow dataset`

In [1]:
import tensorflow_datasets as tfds
import tensorflow as tf
import io

In [2]:
# Load the dataset

imdb, info = tfds.load("imdb_reviews", with_info=True, as_supervised=True, data_dir="./Data/", download=True)
print(f"Dataset info: {info}")

Dataset info: tfds.core.DatasetInfo(
    name='imdb_reviews',
    full_name='imdb_reviews/plain_text/1.0.0',
    description="""
    Large Movie Review Dataset. This is a dataset for binary sentiment
    classification containing substantially more data than previous benchmark
    datasets. We provide a set of 25,000 highly polar movie reviews for training,
    and 25,000 for testing. There is additional unlabeled data for use as well.
    """,
    config_description="""
    Plain text
    """,
    homepage='http://ai.stanford.edu/~amaas/data/sentiment/',
    data_dir='Data/imdb_reviews/plain_text/1.0.0',
    file_format=tfrecord,
    download_size=80.23 MiB,
    dataset_size=129.83 MiB,
    features=FeaturesDict({
        'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
        'text': Text(shape=(), dtype=string),
    }),
    supervised_keys=('text', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <SplitInfo num_examples

2025-07-06 19:11:40.656453: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-07-06 19:11:40.656518: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-07-06 19:11:40.656530: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
I0000 00:00:1751809300.656561 20418091 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1751809300.656609 20418091 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
# Take 4 elements from imdb
for example in imdb['train'].take(1):
    print(f"Type of example: {type(example)}\n")
    print(f"Example data: {example}")

Type of example: <class 'tuple'>

Example data: (<tf.Tensor: shape=(), dtype=string, numpy=b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.">, <tf.Tensor: shape=(), dtype=int64, numpy=0>)


2025-07-06 19:11:40.770290: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2025-07-06 19:11:40.773209: W tensorflow/core/kernels/data/cache_dataset_ops.cc:916] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2025-07-06 19:11:40.773654: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Split the dataset

In [4]:
# Get the training and test sets
train_dataset, test_dataset = imdb['train'], imdb['test']

In [5]:
VOCAB_SIZE = 10000
MAX_LENGTH = 120
EMBEDDING_DIMENSION = 16

### Generate padded sequences

1. Now we are going to pre-proces the data to generate padded sequences (tokenization)
2. We are first going to separate the reviews and labels

In [6]:
# Instantiate the vectorize layer
vectorize_layer = tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=120)

# Separate the reviews and labels
train_reviews = train_dataset.map(lambda review, label: review)
train_labels = train_dataset.map(lambda review, label: label)

test_reviews = test_dataset.map(lambda review, label: review)
test_labels = test_dataset.map(lambda review, label: label)

# Adapt the dataset on the reviews
vectorize_layer.adapt(train_reviews)

2025-07-06 19:15:46.583759: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [7]:
# Explore 10 maximum occurence words
print("Top 10 most occurred words:")
vectorize_layer.get_vocabulary()[:10]

Top 10 most occurred words:


['',
 '[UNK]',
 np.str_('the'),
 np.str_('and'),
 np.str_('a'),
 np.str_('of'),
 np.str_('to'),
 np.str_('is'),
 np.str_('in'),
 np.str_('it')]

#### Post-padding using vectorize layer

In [8]:
# Generate padded sequences from tf.data.Dataset
def vectorize_data(review):
#     Convert the data to tokens
    vectorized_text = vectorize_layer(review)
    return vectorized_text

In [9]:
train_reviews_processed = train_reviews.map(vectorize_data)
test_reviews_processed = test_reviews.map(vectorize_data)

In [10]:
# Get an example from the dataset
for example in train_reviews.take(1):
    print(f"Actual review:\n {example}")
    
for example in train_reviews_processed.take(1):
    print(f"\nProcessed review:\n {example}")

Actual review:
 b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it."

Processed review:
 [  11   14   34  412  384   18   90   28    1    8   33 1322 3560   42
  487    1  191   24   85  152   19   11  217  316   28   65  240  214
    8  489   54   65   85  112   96   22 5596   11   93  642  743   11
   18    7   34  394 9522  170 2464  4

2025-07-06 19:15:46.931098: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [11]:
vectorize_layer.get_vocabulary()[141]

np.str_('through')

#### Combine the vectorized reviews with labels

In [12]:
# Combine
train_dataset_zipped = tf.data.Dataset.zip(train_reviews_processed, train_labels)
test_dataset_zipped = tf.data.Dataset.zip(test_reviews_processed, test_labels)

In [13]:
for example in train_dataset_zipped.take(1):
    print(f"Actual review:\n {example}")

Actual review:
 (<tf.Tensor: shape=(120,), dtype=int64, numpy=
array([  11,   14,   34,  412,  384,   18,   90,   28,    1,    8,   33,
       1322, 3560,   42,  487,    1,  191,   24,   85,  152,   19,   11,
        217,  316,   28,   65,  240,  214,    8,  489,   54,   65,   85,
        112,   96,   22, 5596,   11,   93,  642,  743,   11,   18,    7,
         34,  394, 9522,  170, 2464,  408,    2,   88, 1216,  137,   66,
        144,   51,    2,    1, 7558,   66,  245,   65, 2870,   16,    1,
       2860,    1,    1, 1426, 5050,    3,   40,    1, 1579,   17, 3560,
         14,  158,   19,    4, 1216,  891, 8040,    8,    4,   18,   12,
         14, 4059,    5,   99,  146, 1241,   10,  237,  704,   12,   48,
         24,   93,   39,   11, 7339,  152,   39, 1322,    1,   50,  398,
         10,   96, 1155,  851,  141,    9,    0,    0,    0,    0])>, <tf.Tensor: shape=(), dtype=int64, numpy=0>)
